In [4]:
import fitz  # PyMuPDF
import pandas as pd
import json
import chardet
from time import sleep
import time
import requests
import pandas as pd
import random
import pymysql
import json
import sys
import sqlalchemy
from datetime import datetime,date, timedelta
import os
import openai
from pathlib import Path
from sqlalchemy import text
from datetime import datetime
import shutil
import os
import shutil
import re
import json
from urllib.parse import urlparse, parse_qs
from urllib import parse

def parse_jcurl(url):
    # 解析URL中的查询参数
    parsed_url = urlparse(url)
    query_params = parse_qs(parsed_url.query)
    announceId=int(query_params['announcementId'][0])
    announceTime=query_params['announcementTime'][0]

    url0=f'http://www.cninfo.com.cn/new/announcement/bulletin_detail?announceId={announceId}&flag=true&announceTime={announceTime}'
    hd = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6',
        'Connection': 'keep-alive',
        'Content-Length': '0',
        # 'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
        'Cookie': 'JSESSIONID=32094F939AC5788AE54B85584B1E247E; insert_cookie=45380249; routeId=.uc1; SID=2ba93430-8d4d-47c0-9db5-bceb6cb7e8ba; _sp_ses.2141=*; SF_cookie_4=90357558; _sp_id.2141=6c21141c-7ece-48ea-9323-ac846752aaca.1697523969.13.1720139400.1719994574.b8bf94a9-7bc1-458d-9183-fd36bb5dc6d6',
        'Host': 'www.cninfo.com.cn',
        'Origin': 'http://www.cninfo.com.cn',
        #'Pragma': 'no-cache',
        'Referer': url0,
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 Edg/126.0.0.0',
        }
    r = requests.post(url0,headers=hd,data=data)
    r = str(r.content, encoding="utf-8")
    r = json.loads(r)
    fileurl=r['announcement']['announcementTitle']
    filename=r['fileUrl']
    return fileurl,filename

sql_engine = sqlalchemy.create_engine(
'mysql+pymysql://%s:%s@%s:%s/%s' % (
    'hz_work',
    'Hzinsights2015',
    'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
    '3306',
    'yq',
), poolclass=sqlalchemy.pool.NullPool
)
sql_engine1 = sqlalchemy.create_engine(
'mysql+pymysql://%s:%s@%s:%s/%s' % (
    'root',
    's5bx55dh',
    'bja.sealos.run',
    '44525',
    'timinfo',
), poolclass=sqlalchemy.pool.NullPool
)

destination_folder = r'D:\2024年\小文件\未ocr'
destination_folder1 = r'D:\2024年\小文件\未ocr\1'

def download_PDF(fileUrl,fileName):  #下载pdf
  file_path = os.path.join(destination_folder,fileName+".pdf")
  # 检查文件是否已存在，如果存在则不下载
  if os.path.exists(file_path):
    print(f"{fileName}File already exists!")
  else:
    r = requests.get(fileUrl)
    with open(file_path, "wb") as f:
      f.write(r.content)
    print(f"Downloaded: {fileName}.pdf\n")

def update_database2(originfilename,fileurl,targetfilename):
  sql = """
  UPDATE 23年财报文件
  set fileName= :targetfilename,
  fileUrl= :fileurl
  where fileName = :originfilename
  """
  # 构建参数字典
  params = {
    "originfilename":originfilename,
    "fileurl":fileurl,
    "targetfilename":targetfilename
  }
  with sql_engine.begin() as connection:
    connection.execute(text(sql), params)
def ocr_database(originfilename,beizhu):
  sql = """
  UPDATE 23年财报文件
  set ocr= :beizhu
  where fileName = :originfilename
  """
  # 构建参数字典
  params = {
    "originfilename":originfilename,
    "beizhu":beizhu
  }
  with sql_engine.begin() as connection:
    connection.execute(text(sql), params)


def extract_links(pdf_path):
    doc = fitz.open(pdf_path)
    links = []

    for page in doc:
        annots = page.annots()
        for annot in annots:
            if annot.type[0] == 1:  # 类型为链接
                url = annot.uri  # 提取链接URL
                links.append(url)
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text = page.get_text("text")  # 获取页面文本
        
        # 使用正则表达式匹配URL
        url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
        urls = url_pattern.findall(text)
        
        links.extend(urls)  # 将找到的URL添加到列表中
    return links
def check_pdf_for_images(pdf_path):
    """
    检查PDF文件中是否有需要的文字内容
    """
    doc = fitz.open(pdf_path)  # 打开PDF文件
    has_content = False  # 初始化标记，表示是否找到图片
    pd1=0
    pd2=0
    for page_num in range(doc.page_count):  # 遍历每一页
        page = doc[page_num]
        # 获取页面文本
        page_text1 = page.get_text('blocks')
        # 初始化用于存储合并文本的列表
        page_text_list = []
        # 遍历文本块
        for block in page_text1:
            # 获取文本块的内容
            text = block[4]  # 文本块的内容通常在索引4处
            # 替换块内的 \n 为 \t
            text = text.replace('\n','')
            # 将文本块添加到列表中
            page_text_list.append(text)
            # 将文本块列表合并成一个字符串，每个块之间添加 \n
        page_text = '行开头19910513'.join(page_text_list)
        if '资本公积' in page_text and '附注' in page_text:
            pd1=1
        if '未分配利润' in page_text and '附注' in page_text:
            pd2=1
        
        if pd1==1 and pd2==1:
            has_content = True
            doc.close()
            return has_content
    doc.close()  # 关闭文档
    return has_content

sql="""
SELECT fileName
from yq.23年财报文件
where ocr = '未ocr'"""
with sql_engine.begin() as connection:
    df = pd.read_sql(sql, connection)
fileNamelist=df['fileName'].tolist()
if not os.path.exists(destination_folder1):
    os.makedirs(destination_folder1)

# print(df)
count=1
for filename in os.listdir(destination_folder):
    print(count)
    count+=1
    # ocr_database(filename,'已ocr')
    if filename in fileNamelist:
       ocr_database(filename,'问题')
       continue
    

    # filepath=os.path.join(destination_folder,filename)
    # try:
    #     if check_pdf_for_images(filepath)==False:
    #         ocr_database(filename,'未ocr')
    #         # shutil.copy(filepath,destination_folder1)
    #         # os.remove(filepath)
    #         # print('未ocr')
    #         1
    #     else:
    #         print('已ocr')
    #         ocr_database(filename,'已ocr')
    # except Exception as e:
    #     print(e)
    #     ocr_database(filename,'问题')

      
    

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
